<a href="https://colab.research.google.com/github/abdiToldSo/Bitcamp-2025-Materials/blob/master/aiMLTrack/AI_Workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 ## Workshop: Basic Retrieval-Augmented Generation (RAG)

 This notebook demonstrates a simple RAG pipeline using Google Gemini for LLM responses and Pinecone as the vector database.

 ---

 ### Step 1: Setup and Install Dependencies
 Ensure you have the required libraries installed:
 ```bash
pip install sentence-transformers google-generativeai pinecone PyMuPDF
 ```


In [ ]:
!pip install google-generativeai pinecone PyMuPDF PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 8.9 MB/s eta 0:00:00


In [ ]:
import os
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec
import fitz
from google.generativeai import configure, GenerativeModel
import json


 ---
 ### Step 2: API Keys Setup
Configure the API keys needed to access Google Gemini and Pinecone services.
You'll need to replace these with your own API keys in a real application. We also initialize the
Pinecone client that we'll use to interact with our vector database. <br>
Gemini API KEY: https://aistudio.google.com/app/apikey <br>
Pinecone API KEY: https://www.pinecone.io/ (to create an account if you haven't already) -> API KEYS -> CREATE NEW API KEY


In [ ]:
#insert API keys, configure DB
gemini_api_key = ""
pinecone_api_key = ""
pinecone_env = "us-east-1"  # Example: 'us-east-1'

configure(api_key=gemini_api_key)
pc = Pinecone(
        api_key=pinecone_api_key,
  )

 ---

 ### Step 3: Initialize Vector Database
 We create (or connect to) an index in Pinecone to store and retrieve vector embeddings.

In [ ]:
index_name = "" #name for your database

existing_indexes = [index["name"] for index in pc.list_indexes()]

# create only if it doesn't exist already
if index_name not in existing_indexes:
  pc.create_index(
      name=index_name,
      dimension= , # Replace with your model dimensions (this embedding model has 384 dimensions)
      metric= , # Replace with your model metric
      spec=ServerlessSpec(
          cloud="aws",
          region="us-east-1"
      )
  )

In [ ]:
index = pc.Index(index_name)
index

 ---

 ### Step 4: Embedding Model Setup
 We use OpenAI's `tall-MiniLM-L6-v2` model to embed documents and queries, which has 384 dimensions. This will very well be different from the embedding dimensions of the Gemini model, but we don't need to consider that since RAG is a standalone process that does not involve the LLM until the final context has been received

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

def embed_text(text):
    embeddings = model.encode(text)  # The model expects a list of texts
    return embeddings

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 ---

 ### Step 5: Ingest Documents into Pinecone
 Convert the extracted text from the PDF into embeddings and store them in Pinecone.
 These are helper methods to get your pdf embedded and stored in a vectorDB. Big pdfs can take a bit of time.

In [ ]:
from PyPDF2 import PdfReader

def extract_text_from_pdf(pdf_file_path):
    # Extract text from the entire PDF document
    pdf_text = ""
    with open(pdf_file_path, "rb") as file:
        reader = PdfReader(file)
        for page in reader.pages:
            pdf_text += page.extract_text()
    return pdf_text

def chunk_text(text, chunk_size=500):
    # Split text into smaller chunks based on the chunk_size (e.g., 500 characters)
    # This is a simple approach, there are ways to do better (having overlap, etc)
    chunks = []
    print("Total Chunks to be processed: ", len(text)//chunk_size)
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i+chunk_size])
    return chunks

def ingest_pdf(pdf_file_path, doc_id):
    print("Starting Text Extraction...")
    pdf_text = extract_text_from_pdf(pdf_file_path)
    print("Starting Chunking ...")
    text_chunks = chunk_text(pdf_text, chunk_size = ) #mention chunk size
    print("Starting the Embedding process ...")
     # Generate embeddings for all text chunks at once
    embeddings = embed_text(text_chunks)
    print(embeddings.shape)
    print("Data prepared to upsert")
    # Prepare upsert payload
    upsert_data = [
        (f"{doc_id}_{i}", embedding.tolist(), {"text": chunk})
        for i, (embedding, chunk) in enumerate(zip(embeddings, text_chunks))
    ]
    # Upsert all at once
    index.upsert(upsert_data)
    print("data upserted :)")


In [ ]:
# Ingest an uploaded PDF document:
ingest_pdf("path/to/uploaded_pdf", "doc1") #doc1 is a sample identifier, you can make it anything

Starting Text Extraction...
Starting Chunking ...
Total Chunks to be processed:  61
Starting the Embedding process ...
(62, 384)
Data prepared to upsert


 ---

 ### Step 6: Retrieve Relevant Documents
 Given a query, retrieve the most relevant document using similarity search. We use all the functions already coded in the notebook to query the vector db

In [ ]:
def retrieve_relevant_docs(query, top_k=5):
    query_embedding = embed_text([query]) #the function expects a list of vectors
    query_embedding = query_embedding.tolist()  # Convert ndarray to list

    # Query Pinecone for the top_k most relevant chunks
    results = index.query(vector=query_embedding, top_k=top_k, include_metadata=True)

    # Return the relevant text chunks
    return [match["metadata"]["text"].replace("\n","") for match in results["matches"]]

In [ ]:
query = "" #user query
retrieved_docs = retrieve_relevant_docs(query)

In [ ]:
retrieved_docs

['ing  class  with the  instructor’s  or peer mentor’s  assistance  as needed.  You will submit  your work  individually  via ELMS and  will receive  a completion  grade.  Most  lab exercises  will take place  on Thursdays  and many  will be done  with your project  team  to foster  team  cohesion.  On discussion/lab  days,  attendance  is required for  full credit  on the lab exercise.  Sign-in logs will be used to track  attendance.  [F/I] ● Quizzes (5% of final grade):  Quizzes  will test ',
 '  work.   ● Lectures:  You are expected  to complete  the assigned  reading  before  class  time.  Lectures  will be interactive.    4 Participation is necessary in order to  excel at this course. If you anticipate missing a session, email the instructors  and TA ahead  of time,  or contact  us soon  afterwards.  At least three  "participation  instances"  per week is advised and  encouraged.  ● Discussions (5% of final grade):  Biweekly,  asynchronous  online  discussions  will test your ',
 


 ---

 ### Step 7: Generate Response using Google Gemini
 We use the retrieved context to generate a response with the LLM.

In [ ]:
gemini_model = GenerativeModel(model_name="gemini-2.5-pro-exp-03-25")

def generate_answer(query, context):
    prompt1 = f"Use ONLY the context provided to answer the query. Context: {context}\n\nQuestion: {query}\n\nAnswer:"
    response1 = gemini_model.generate_content(prompt1)
    prompt2 = f"answer this question {query}"
    response2=  gemini_model.generate_content(prompt2)
    return response1.text, response2.text

In [ ]:
answer1, answer2 = generate_answer(query, retrieved_docs)

 ---

 ### Step 8: Display Results
 We print the retrieved context and the final response.

In [ ]:
print("Query:", query)
print("\nRetrieved Context:", retrieved_docs[0])
print("\nRAG Answer:", answer1)

Query: Are there some easy ways to get an A in the class

Retrieved Context: ing  class  with the  instructor’s  or peer mentor’s  assistance  as needed.  You will submit  your work  individually  via ELMS and  will receive  a completion  grade.  Most  lab exercises  will take place  on Thursdays  and many  will be done  with your project  team  to foster  team  cohesion.  On discussion/lab  days,  attendance  is required for  full credit  on the lab exercise.  Sign-in logs will be used to track  attendance.  [F/I] ● Quizzes (5% of final grade):  Quizzes  will test 

RAG Answer: Based on the provided context, there is no mention of "easy ways" to get an A. The text indicates that excelling in the course requires participation, attendance (which is described as crucial), completing assigned readings, contributing substantively to discussions, and completing lab exercises.


In [ ]:
print("\nNormal Answer:", answer2)


Normal Answer: Okay, let's talk about getting an 'A'. While there's rarely a magic bullet or a way to get an 'A' with *zero* effort, there are definitely **smart and efficient strategies** that can feel "easier" because they focus your effort effectively and prevent common pitfalls.

Think of these less as "easy ways" and more as **foundational, high-impact actions**:

1.  **Actually Read and Understand the Syllabus:** This is surprisingly often skipped! It's your roadmap.
    *   **Know the Grading Breakdown:** Where do the points come from? Is participation 5% or 25%? Are exams worth more than homework? Knowing this tells you where to focus your energy. Spending hours perfecting homework worth 10% while bombing exams worth 60% isn't efficient.
    *   **Note ALL Deadlines:** Put them in a calendar immediately. Late penalties kill grades.
    *   **Understand Policies:** Attendance, late work, extra credit opportunities.

2.  **Show Up (and Pay Attention):** Physically being in class

In [ ]:
index.delete(deleteAll=True)

{}